In [0]:
# Imports 

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("SilverToGold").getOrCreate()

In [0]:
# Read data

silver_path = (
    "abfss://lakehouse@staqilakehouseen.dfs.core.windows.net/"
    "silver/air_quality/"
)

cleaned_df = spark.read.parquet(silver_path)

cleaned_df = cleaned_df.withColumn(
    "measurement_date",
    F.col("measurement_timestamp").cast("date")
)

In [0]:
# Daily city aqi

daily_city_aqi_df = cleaned_df.groupBy(
    "city",
    "measurement_date"
).agg(
    F.round(F.avg("european_aqi"), 2).alias("avg_european_aqi"),
    F.max("european_aqi").alias("max_european_aqi"),
    F.round(F.avg("pm10"), 2).alias("avg_pm10"),
    F.max("pm10").alias("max_pm10"),
    F.round(F.avg("pm2_5")).alias("avg_pm2_5"),
    F.max("pm2_5").alias("max_pm2_5"),
    F.round(F.avg("nitrogen_dioxide")).alias("avg_nitrogen_dioxide"),
    F.max("nitrogen_dioxide").alias("max_nitrogen_dioxide"),
    F.count("*").alias("record_count")
).orderBy(
    "city",
    "measurement_date"
)

daily_city_aqi_df.show(truncate=False)

+------------+----------------+----------------+----------------+--------+--------+---------+---------+--------------------+--------------------+------------+
|city        |measurement_date|avg_european_aqi|max_european_aqi|avg_pm10|max_pm10|avg_pm2_5|max_pm2_5|avg_nitrogen_dioxide|max_nitrogen_dioxide|record_count|
+------------+----------------+----------------+----------------+--------+--------+---------+---------+--------------------+--------------------+------------+
|Athens      |2026-07-09      |39.67           |60              |18.4    |25.2    |13.0     |17.5     |38.0                |71.2                |18          |
|Berlin      |2026-07-09      |22.0            |30              |8.57    |11.0    |4.0      |4.1      |4.0                 |6.6                 |18          |
|London      |2026-07-09      |32.22           |63              |14.96   |16.9    |10.0     |10.6     |22.0                |39.9                |18          |
|Paris       |2026-07-09      |27.78          

In [0]:
# Daily city ranking by aqi

daily_city_aqi_rank_df = daily_city_aqi_df.withColumn(
    "rank",
    F.rank().over(Window.partitionBy(
        "city"
    ).orderBy(
        F.col("avg_european_aqi").desc()
    )
))

daily_city_aqi_rank_df.show(truncate=False)

+------------+----------------+----------------+----------------+--------+--------+---------+---------+--------------------+--------------------+------------+----+
|city        |measurement_date|avg_european_aqi|max_european_aqi|avg_pm10|max_pm10|avg_pm2_5|max_pm2_5|avg_nitrogen_dioxide|max_nitrogen_dioxide|record_count|rank|
+------------+----------------+----------------+----------------+--------+--------+---------+---------+--------------------+--------------------+------------+----+
|Athens      |2026-07-09      |39.67           |60              |18.4    |25.2    |13.0     |17.5     |38.0                |71.2                |18          |1   |
|Berlin      |2026-07-09      |22.0            |30              |8.57    |11.0    |4.0      |4.1      |4.0                 |6.6                 |18          |1   |
|London      |2026-07-09      |32.22           |63              |14.96   |16.9    |10.0     |10.6     |22.0                |39.9                |18          |1   |
|Paris       |20

In [0]:
# Data freshness

data_freshness_df = cleaned_df.groupBy(
    "city"
).agg(
    F.max("ingestion_timestamp_utc").alias("latest_ingestion_timestamp_utc"),
    F.max("measurement_timestamp").alias("latest_measurement_timestamp")
    )

data_freshness_df.show(truncate=False)

+------------+------------------------------+----------------------------+
|city        |latest_ingestion_timestamp_utc|latest_measurement_timestamp|
+------------+------------------------------+----------------------------+
|Thessaloniki|2026-07-09 14:31:34.85177     |2026-07-09 17:00:00         |
|London      |2026-07-09 14:31:35.437359    |2026-07-09 17:00:00         |
|Paris       |2026-07-09 14:31:35.969701    |2026-07-09 17:00:00         |
|Athens      |2026-07-09 14:31:34.265575    |2026-07-09 17:00:00         |
|Berlin      |2026-07-09 14:31:36.441669    |2026-07-09 17:00:00         |
+------------+------------------------------+----------------------------+



In [0]:
# Data completeness

data_completeness_df = cleaned_df.groupBy(
        "city",
        "measurement_date"
    ).agg(
        F.count("*").alias("actual_records")
    ).withColumn("expected_records", F.lit(24)).withColumn(
        "completeness_pct", F.round(F.col("actual_records") / F.col("expected_records") * 100, 2))
    
data_completeness_df.show(truncate=False)

+------------+----------------+--------------+----------------+----------------+
|city        |measurement_date|actual_records|expected_records|completeness_pct|
+------------+----------------+--------------+----------------+----------------+
|Athens      |2026-07-09      |18            |24              |75.0            |
|Thessaloniki|2026-07-09      |18            |24              |75.0            |
|Paris       |2026-07-09      |18            |24              |75.0            |
|London      |2026-07-09      |18            |24              |75.0            |
|Berlin      |2026-07-09      |18            |24              |75.0            |
+------------+----------------+--------------+----------------+----------------+



In [0]:
gold_base_path = (
    "abfss://lakehouse@staqilakehouseen.dfs.core.windows.net/"
    "gold/"
)

daily_city_aqi_df.write.mode("overwrite").parquet(
    gold_base_path + "daily_city_aqi/"
)

daily_city_aqi_rank_df.write.mode("overwrite").parquet(
    gold_base_path + "daily_city_ranking/"
)

data_freshness_df.write.mode("overwrite").parquet(
    gold_base_path + "data_freshness/"
)

data_completeness_df.write.mode("overwrite").parquet(
    gold_base_path + "data_completeness/"
)